# From Vibes to a Number: One Turn of the Quality Flywheel

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 21 §§21.1–21.4 · type: concept-lab*

**The promise:** you'll stop judging an agent by a single run, learn to see it as a *distribution* of behaviors, write down what "good" means as checkable criteria, and turn the flywheel once — instrument → observe → evaluate → improve — watching a real number move.

Fully offline. No API key, no services, no cost. Everything here is seeded and deterministic so your run matches the book's.

## 🧠 Why this matters

Traditional software is deterministic: if the test passes, the behavior you tested is the behavior you ship. An agent is not like that — the same prompt can yield different outputs, and a "fix" can silently break five other cases. So an agent isn't a *function* that maps an input to an output; it's a **distribution** of behaviors. Run it a hundred times on a hundred realistic inputs and you get a spread: mostly good, some mediocre, a few alarming.

Quality is a property of *that distribution* — its center, its variance, and especially its **tail** — and you cannot perceive a distribution by looking at three cherry-picked examples in a demo. "It worked when I tried it" is one sample drawn from the friendliest corner of the input space, usually by the person most motivated to see it succeed. That's an anecdote, not a measurement. This notebook makes that physical, then shows the loop that turns the anecdote into a trend you can recompute.

## Objectives & prerequisites

**By the end you can:**
- Describe an agent as a *distribution* and sample it instead of trusting one run.
- Write layered success criteria — hard **constraints**, **task success**, **quality gradients** (§21.3).
- Run one full **instrument → observe → evaluate → improve** rotation that demonstrably moves a score (§21.2).
- Recognize *shipping on vibes* and the gap between a cherry-picked demo and the sampled spread.

**Prereqs:** none beyond reading Ch 1–3 and Ch 21. This is the first notebook of Part VI; it sets up the harness vocabulary that Ch 22 turns into real code and Ch 23 turns into real tracing. **Run order:** just run the cells top to bottom.

## Setup

No model calls happen in this notebook — it is offline by design — but we keep the same `MOCK` switch every companion notebook uses, so the habit carries to chapters that *do* call a model. `MOCK=1` (the default) means "canned, deterministic behavior, no key, no cost." Here it stays `1`; later notebooks let you flip it to `0` to hit a live API.

In [ ]:
import os
import json
import random
import statistics
from dataclasses import dataclass, field, asdict

# Load secrets from a local .env if one exists. This notebook needs no key
# (everything is offline/mock), but we keep the habit for chapters that do.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # missing package or no .env — harmless for this offline notebook.

# The standard companion switch. Default 1 = canned/offline/free.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# One seed for the whole notebook so your numbers match the book's exactly.
SEED = 21
random.seed(SEED)

print(f"MOCK mode: {MOCK}  (this notebook is fully offline regardless)")
print(f"Seed: {SEED}")
print("No API key required. Nothing here spends money or hits the network.")

## A toy support agent (and its hidden failure modes)

We need something that *behaves like* a real agent — noisy, mostly-right, occasionally dangerous — without any model or key. So we hand-roll a **mocked** support agent whose canned outputs carry built-in failure modes at *known rates*:

- it sometimes **fabricates a refund promise** it has no authority to make,
- it sometimes makes an **ungrounded claim** (states a policy that isn't in its context),
- it sometimes returns **malformed JSON** (a hard constraint violation downstream),
- and the failure rates are **higher on edge-case inputs** than on the friendly ones — exactly the asymmetry a four-query demo hides.

Because we planted the rates, we know the ground truth and can check whether our eval actually catches them.

In [ ]:
@dataclass
class Trace:
    """A minimal per-run trace — a hand-rolled foreshadow of Ch 23's OTel spans."""
    input: str
    difficulty: str            # "easy" or "edge"
    steps: list = field(default_factory=list)
    output: str = ""
    output_json: str = ""      # the structured field the downstream system parses
    latency_ms: int = 0
    est_cost_usd: float = 0.0
    # Ground-truth flags the mock plants so we can check our eval against reality.
    _planted: dict = field(default_factory=dict)


def mock_support_agent(text, difficulty, rng, system_prompt):
    """A seeded, fully-canned 'agent'. No model, no network.

    Failure rates depend on difficulty AND on the system prompt: a stricter
    prompt (the fix we apply later) suppresses the fabricated-refund failure.
    """
    forbids_refund = "NEVER promise a refund" in system_prompt

    base = 0.08 if difficulty == "easy" else 0.42  # edge cases fail far more often
    p_refund = 0.0 if forbids_refund else base
    p_ungrounded = base * 0.6
    p_badjson = 0.02 if difficulty == "easy" else 0.10

    fabricated_refund = rng.random() < p_refund
    ungrounded_claim = rng.random() < p_ungrounded
    bad_json = rng.random() < p_badjson

    steps = ["classify_intent", "lookup_policy", "draft_reply"]
    reply = "Thanks for reaching out — here's how we can help with your issue."
    if fabricated_refund:
        reply += " I've gone ahead and issued you a full refund."
    if ungrounded_claim:
        reply += " Our policy guarantees same-day replacement worldwide."

    resolved = not (fabricated_refund or ungrounded_claim) and rng.random() < 0.9
    payload = {"reply": reply, "resolved": resolved, "escalate": not resolved}
    output_json = "{not valid json" if bad_json else json.dumps(payload)

    return Trace(
        input=text,
        difficulty=difficulty,
        steps=steps,
        output=reply,
        output_json=output_json,
        latency_ms=rng.randint(380, 1600),
        est_cost_usd=round(rng.uniform(0.002, 0.011), 4),
        _planted={
            "fabricated_refund": fabricated_refund,
            "ungrounded_claim": ungrounded_claim,
            "bad_json": bad_json,
            "resolved": resolved,
        },
    )


# The original, loose system prompt — the one shipped "on vibes."
SYSTEM_PROMPT_V1 = "You are a friendly support agent. Be helpful and resolve the issue."
print("Mock agent ready. No model, no key — just seeded RNG with planted failure modes.")

### The demo that looks great

Here's how almost every agent gets blessed for production: someone runs it once on a friendly input, reads a nice answer, and nods. Let's do exactly that.

In [ ]:
demo_rng = random.Random(SEED)  # isolated stream so the demo is reproducible
demo = mock_support_agent(
    "Hi, how do I update the email address on my account?",
    difficulty="easy",
    rng=demo_rng,
    system_prompt=SYSTEM_PROMPT_V1,
)
print("INPUT :", demo.input)
print("OUTPUT:", demo.output)
print("JSON  :", demo.output_json)
print("\nLooks great. Ship it, right?")

### 🔮 Predict

That one run looked perfect. Before we sample more, **write down your guess**: across **100** realistic inputs — a *mix* of easy questions and trickier edge cases (refund disputes, angry tone, ambiguous requests) — what fraction do you think will pass a strict quality bar (no fabricated refund, no ungrounded claim, valid JSON, issue resolved)?

Jot a number. 95%? 80%? Then run the next cells. The gap between your guess and the measured value *is* the lesson.

## Sample the distribution

First we need realistic inputs. We generate a tiny in-notebook set of ~12 seed inputs across both difficulty bands, then expand to 100 by sampling them — still entirely offline, nothing external.

In [ ]:
EASY_INPUTS = [
    "How do I update the email on my account?",
    "Where can I download my invoice?",
    "What are your support hours?",
    "How do I reset my password?",
    "Can I change my plan from monthly to yearly?",
    "How do I add a teammate to my workspace?",
]
EDGE_INPUTS = [
    "This is the third time it broke. I want my money back NOW.",
    "Your product deleted my data and I demand compensation.",
    "I was charged twice and nobody has helped me for a week.",
    "Cancel everything and refund all 14 months or I'm calling my lawyer.",
    "It doesn't work. Just fix it. (no other details)",
    "Do you guarantee delivery to Antarctica by tomorrow?",
]


def build_eval_set(n, rng):
    """~half easy, ~half edge — a realistic spread, not the friendliest corner."""
    items = []
    for _ in range(n):
        if rng.random() < 0.5:
            items.append((rng.choice(EASY_INPUTS), "easy"))
        else:
            items.append((rng.choice(EDGE_INPUTS), "edge"))
    return items


eval_rng = random.Random(SEED + 1)
eval_set = build_eval_set(100, eval_rng)
n_easy = sum(1 for _, d in eval_set if d == "easy")
print(f"Built {len(eval_set)} realistic inputs: {n_easy} easy / {len(eval_set) - n_easy} edge.")

## Defining "good" as layered criteria (§21.3)

You cannot measure quality until you define it. The discipline is to translate fuzzy intent into **checkable predicates** over a run. We decompose "good" into three layers, exactly as the chapter does:

1. **Hard constraints** — must *never* fail; these **gate** a release. (No fabricated refund, no ungrounded claim, valid JSON.)
2. **Task success** — the user's actual goal was achieved (issue resolved without escalation).
3. **Quality gradients** — nice-to-haves scored, not gated (concise, grounded tone).

Each becomes a small function that reads a `Trace` and returns a verdict. *Writing these functions is the product decision* — more on that in the senior lens.

In [ ]:
def check_constraints(trace):
    """Hard gates. Returns a dict of {constraint_name: passed?}."""
    # Valid JSON is a code-checkable constraint.
    try:
        json.loads(trace.output_json)
        valid_json = True
    except (json.JSONDecodeError, TypeError):
        valid_json = False
    no_fabricated_refund = "issued you a full refund" not in trace.output
    no_ungrounded_claim = "same-day replacement worldwide" not in trace.output
    return {
        "valid_json": valid_json,
        "no_fabricated_refund": no_fabricated_refund,
        "no_ungrounded_claim": no_ungrounded_claim,
    }


def check_task_success(trace):
    """Outcome: was the issue resolved without escalation? (read from the structured field)"""
    try:
        payload = json.loads(trace.output_json)
    except (json.JSONDecodeError, TypeError):
        return False  # can't even parse it — not a success
    return bool(payload.get("resolved")) and not payload.get("escalate")


def check_gradients(trace):
    """Soft quality scored 0..1 (here: conciseness). Not a gate."""
    words = len(trace.output.split())
    return 1.0 if words <= 22 else max(0.0, 1.0 - (words - 22) / 40)


def passed_strict(trace):
    """The strict bar: ALL constraints hold AND the task succeeded."""
    return all(check_constraints(trace).values()) and check_task_success(trace)


print("Criteria defined: 3 hard constraints (gate), 1 task-success check, 1 gradient (scored).")

### 🔧 Instrument: run the suite and emit a trace per run

*Instrument* means every run emits a trace — input, steps, output, latency, cost — before you know what you'll need it for. We do the bare-minimum version by hand; Ch 23 replaces this with real OpenTelemetry spans. Here we run all 100 inputs through V1 of the agent and keep every trace.

In [ ]:
def run_suite(eval_set, system_prompt, seed):
    """Run every input, return the list of traces. Seeded => reproducible."""
    rng = random.Random(seed)
    return [
        mock_support_agent(text, diff, rng, system_prompt)
        for text, diff in eval_set
    ]


def score_suite(traces):
    """Aggregate the distribution into the numbers a senior actually reports."""
    n = len(traces)
    strict_pass = sum(passed_strict(t) for t in traces) / n
    # Per-constraint violation tail — the alarming part a demo hides.
    refund_violations = sum(not check_constraints(t)["no_fabricated_refund"] for t in traces)
    ungrounded = sum(not check_constraints(t)["no_ungrounded_claim"] for t in traces)
    bad_json = sum(not check_constraints(t)["valid_json"] for t in traces)
    return {
        "n": n,
        "strict_pass_rate": round(strict_pass, 3),
        "task_success_rate": round(sum(check_task_success(t) for t in traces) / n, 3),
        "mean_gradient": round(statistics.mean(check_gradients(t) for t in traces), 3),
        "p95_latency_ms": sorted(t.latency_ms for t in traces)[int(0.95 * n) - 1],
        "total_cost_usd": round(sum(t.est_cost_usd for t in traces), 4),
        "tail": {
            "fabricated_refund": refund_violations,
            "ungrounded_claim": ungrounded,
            "invalid_json": bad_json,
        },
    }


traces_v1 = run_suite(eval_set, SYSTEM_PROMPT_V1, seed=SEED + 2)
score_v1 = score_suite(traces_v1)
print("V1 (shipped on vibes) — the whole distribution, not one run:")
print(json.dumps(score_v1, indent=2))

### What you just saw

Compare the **strict pass rate** to the number you predicted. The single demo run looked perfect, but across the real spread a meaningful slice fails — and the `tail` shows *why*: the agent is **confidently wrong** (fabricated refunds, ungrounded claims) on the edge cases the demo never touched. Let's make the spread visible with a quick text histogram — no plotting library, just the distribution split by difficulty.

In [ ]:
def histogram_by_difficulty(traces, label):
    print(f"Strict-pass distribution — {label}")
    for diff in ("easy", "edge"):
        subset = [t for t in traces if t.difficulty == diff]
        if not subset:
            continue
        rate = sum(passed_strict(t) for t in subset) / len(subset)
        bar = "█" * round(rate * 30)
        print(f"  {diff:<5} n={len(subset):<3} {bar:<30} {rate:5.0%}")
    overall = sum(passed_strict(t) for t in traces) / len(traces)
    print(f"  {'ALL':<5} n={len(traces):<3} {'█' * round(overall * 30):<30} {overall:5.0%}")


histogram_by_difficulty(traces_v1, "V1")
print("\nThe 'friendly corner' (easy) is where the demo lived. The edge tail is the real risk.")

### ⚠️ Pitfall: shipping on vibes

This is the most expensive failure mode in agentic engineering. The demo looked great, the PM tried four queries, everyone nodded, deploy. Let's *reproduce* the trap: cherry-pick four easy inputs (the demo) and "measure" on those — then look at the honest sampled number next to it.

In [ ]:
cherry_rng = random.Random(SEED)
cherry_inputs = [(random.choice(EASY_INPUTS), "easy") for _ in range(4)]
cherry_traces = run_suite(cherry_inputs, SYSTEM_PROMPT_V1, seed=SEED)
cherry_rate = sum(passed_strict(t) for t in cherry_traces) / len(cherry_traces)

print(f"Demo  (4 cherry-picked easy queries): {cherry_rate:.0%} pass  → 'looks done!'")
print(f"Truth (100 sampled realistic inputs): {score_v1['strict_pass_rate']:.0%} pass")
print(f"\nThe gap ({cherry_rate - score_v1['strict_pass_rate']:+.0%}) is the bug report you'll get in week 3.")

## Observe → Evaluate: read traces, build an error taxonomy, make a golden set

No dashboard replaces *looking at the data*. The single highest-leverage habit in applied AI is reading real traces end to end, labelling failures, and sorting them into buckets. We read a handful of failing traces, bucket them into an **error taxonomy**, then promote the failures into a tiny **golden set** with tags — the seed of an asset that, six months in, no competitor can copy.

In [ ]:
def classify_failure(trace):
    """Bucket one failing run into the error taxonomy (observe → evaluate)."""
    c = check_constraints(trace)
    if not c["no_fabricated_refund"]:
        return "fabricated_refund"      # the dangerous one
    if not c["no_ungrounded_claim"]:
        return "ungrounded_claim"
    if not c["valid_json"]:
        return "invalid_json"
    if not check_task_success(trace):
        return "unresolved"
    return "ok"


failures = [t for t in traces_v1 if not passed_strict(t)]
print(f"Reading the first few of {len(failures)} failing traces:\n")
for t in failures[:3]:
    print(f"  [{t.difficulty}] bucket={classify_failure(t)!r:20} input={t.input[:48]!r}")

# Bucket the whole failure set into a taxonomy.
taxonomy = {}
for t in failures:
    taxonomy[classify_failure(t)] = taxonomy.get(classify_failure(t), 0) + 1
print("\nError taxonomy (your prioritized backlog):")
for bucket, count in sorted(taxonomy.items(), key=lambda kv: -kv[1]):
    print(f"  {bucket:<20} {count}")

In [ ]:
# Promote failures into a tagged golden set — the durable asset (previews
# templates/eval-dataset-template/, fleshed out in Ch 22).
golden_set = [
    {"input": t.input, "difficulty": t.difficulty, "failure_tag": classify_failure(t)}
    for t in failures
]
print(f"Golden set seeded with {len(golden_set)} real failures, tagged by bucket.")
print("Sample cases:")
print(json.dumps(golden_set[:3], indent=2))
print("\nEvery interesting production failure becomes a permanent eval case.")

## Improve: apply one fix, re-run, watch the number (and the tail) move

The taxonomy says the top — and most dangerous — bucket is **fabricated refunds**. So we make exactly one change: tighten the system prompt to forbid that commitment. Then we re-run the *same* suite and let the eval be the referee. This is the flywheel's edge from *observe* back into *evaluate*, made concrete.

In [ ]:
# The fix: one targeted constraint added to the prompt.
SYSTEM_PROMPT_V2 = (
    SYSTEM_PROMPT_V1
    + " NEVER promise a refund or any compensation; escalate refund requests to a human."
)

traces_v2 = run_suite(eval_set, SYSTEM_PROMPT_V2, seed=SEED + 2)
score_v2 = score_suite(traces_v2)

print("V2 (after one targeted fix):")
print(json.dumps(score_v2, indent=2))

In [ ]:
# The eval diff — the only artifact that should gate a ship.
def pct(x):
    return f"{x:.0%}"


print("EVAL DIFF  V1 → V2  (neutral-or-better is the bar)")
print(f"  strict pass rate : {pct(score_v1['strict_pass_rate'])} → {pct(score_v2['strict_pass_rate'])}  "
      f"({score_v2['strict_pass_rate'] - score_v1['strict_pass_rate']:+.0%})")
print(f"  task success     : {pct(score_v1['task_success_rate'])} → {pct(score_v2['task_success_rate'])}")
print("  tail — fabricated refunds:",
      score_v1['tail']['fabricated_refund'], "→", score_v2['tail']['fabricated_refund'])
print("  tail — ungrounded claims :",
      score_v1['tail']['ungrounded_claim'], "→", score_v2['tail']['ungrounded_claim'])
print()
histogram_by_difficulty(traces_v2, "V2")

### What you just saw

One prompt change, and the **measured score moved** — with the most dangerous tail bucket (fabricated refunds) driven to zero, while task success didn't regress. That sentence — "V2 scores higher on the golden set with no regression on the safety slice" — is what replaces "the new prompt feels better." You just turned the flywheel one full rotation: **instrument** (traces) → **observe** (read failures) → **evaluate** (taxonomy + golden set + score) → **improve** (the fix, gated by the eval diff). The next rotation starts the moment V2 ships back into the instrumented system.

## 🎯 Senior lens

Notice what was actually hard here. It wasn't the fix — that was one sentence. It was **deciding what "good" means**: *is refunding the customer without asking a success or an incident?* Writing `check_constraints` and `check_task_success` *is the product decision*, and it cannot be delegated to the model. Senior engineers treat eval design as **requirements engineering**: they interrogate stakeholders, encode the answers as checkable predicates, and version them like code.

The second judgment call: the **eval suite, not the prompt, is the durable asset.** Prompts and models get swapped many times; the golden set distilled from *your* users' real failures outlives all of them. The team that owns the sharpest definition of "good" ships the best agent — almost independently of whose prompts are cleverer. So spend your scarce attention on the criteria and the golden set, not on prompt micro-tuning you can't measure.

## Recap

- An agent is a **distribution** of behaviors, not a function. Judge it by *sampling* — many realistic inputs — and read the center, the variance, and especially the **tail**.
- A single demo run is the friendliest corner of the input space. **Shipping on vibes** hides a measurable slice of confidently-wrong behavior.
- "Good" decomposes into **constraints** (gate), **task success** (golden set), and **quality gradients** (scored). Writing these predicates is the product decision.
- The **quality flywheel** — instrument → observe → evaluate → improve — turns anecdotes into a number with a trend. The edge from *observe* back into *evaluate* (failures become permanent eval cases) is what compounds.
- The eval suite, not the prompt, is the asset competitors can't copy.

## 📋 Day-one quality checklist (§21.4)

Install these habits *before* your first agent reaches production:

- [ ] Every agent run emits a trace with inputs, steps, outputs, latency, and cost.
- [ ] "Good" is written down as checkable criteria: constraints, task success, gradients.
- [ ] A golden set exists — even twenty cases — and runs on every behavior-changing diff.
- [ ] Production failures are triaged into an error taxonomy and feed the eval suite.
- [ ] A recurring error-analysis session is on the calendar with traces on the screen.
- [ ] One named person owns the eval suite and the quality trend line.
- [ ] No prompt, tool, or model change ships without an eval comparison against baseline.

## Exercises

Each task *changes* something and asks you to **predict before you measure**. Solutions live in `solutions/` (added in Phase 2), not inline.

1. **Add a failure mode and a criterion to catch it.** Extend `mock_support_agent` so it sometimes **leaks PII** (e.g. echoes a fake account number), then add a `no_pii_leak` hard constraint to `check_constraints`. 🔮 Predict the new strict pass rate before re-running, then measure it.
2. **Shift the input mix.** Change `build_eval_set` to 80% edge / 20% easy. 🔮 Predict how the V1 strict pass rate moves, then confirm. What does this say about how representative your eval set must be?
3. **A fix that helps one thing and hurts another.** Write a `SYSTEM_PROMPT_V3` that makes replies much longer/more cautious. 🔮 Predict the effect on the `mean_gradient` (conciseness) *and* on task success, then measure. Decide from the eval diff whether you'd ship it.
4. **Grow the golden set.** Run V2, collect its *remaining* failures, and merge them into `golden_set` with tags. How many rotations until the tail empties? What does that tell you about when an eval suite is "done"?

In [ ]:
# Exercise 1 — add a PII-leak failure mode + a no_pii_leak constraint, then predict & measure.


In [ ]:
# Exercise 2 — shift build_eval_set to 80% edge / 20% easy; predict then re-score V1.


In [ ]:
# Exercise 3 — SYSTEM_PROMPT_V3 (longer/cautious); predict gradient vs task-success trade-off.


In [ ]:
# Exercise 4 — collect V2's remaining failures and grow the golden set; count rotations to empty tail.


## Next

You ran the flywheel by hand. The next two chapters build the *real* stations:

- **Ch 22 — Evaluation** turns this notebook's ad-hoc `check_*` predicates and golden-set sketch into a real eval harness and a CI gate. It previews [`blueprints/eval-harness/`](../../../blueprints/eval-harness/) and [`templates/eval-dataset-template/`](../../../templates/eval-dataset-template/), and lands in `capstone/evals/`.
- **Ch 23 — Observability** replaces our hand-rolled `Trace` with real OpenTelemetry spans, feeding [`blueprints/observability-stack/`](../../../blueprints/observability-stack/) and `capstone/telemetry.py`.

Carry one thing forward: *no change ships without an eval diff.* Everything in Part VI exists to make that diff trustworthy.